In [ ]:
# ========== 导入 + 环境 + 客户端 + 模型常量 ==========

# 导入标准库 os：读环境变量（Environment Variables）
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端类：调用 Chat Completions API
from openai import OpenAI
# 从 IPython.display 导入展示工具：在笔记本里漂亮显示 Markdown
from IPython.display import Markdown, display

# 加载 .env：override=True 表示用文件值覆盖已有环境变量
load_dotenv(override=True)
# 读取 API Key（键名必须是 OPENAI_API_KEY）
api_key = os.getenv('OPENAI_API_KEY')

# ---------- 密钥自检（打印文案保持原样） ----------
if not api_key:
    print("No API key was found!")
elif not api_key.startswith("sk-proj-"):
    print("Wrong API key format!")
elif api_key.strip() != api_key:
    print("API key has extra spaces!")
else:
    print("API key found and looks good!")

# 创建默认 OpenAI 客户端（自动使用环境变量里的密钥）
openai = OpenAI()

# 模型常量：集中写名字，后面四个函数都引用它
MODEL_GPT = 'gpt-4o-mini'


In [ ]:
# ========== 步骤 1：System Prompt（邮件助手的总规则） ==========

# system_prompt 保留英文：发给模型的指令，翻译会改变行为
# 这里声明助手能做四件事：摘要 / 主题行 / 翻译摘要 / 专业回复
system_prompt = """You are a helpful email assistant.
You can do the following tasks:
1. Summarize emails
2. Suggest subject lines
3. Translate email summaries 
4. Reply to emails professionally
Please format your response in Markdown."""


In [ ]:
# ========== 步骤 2：四个邮件能力函数（都走同一套 Chat Completions 模式） ==========

# --- 能力 1：摘要邮件 ---
def summarize_email(email_content):
    """Summarize the email content"""
    # messages：system 定身份，user 放「请摘要」+ 邮件正文
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"""
        Please summarize this email in a few sentences:
        
        {email_content}
        """}
    ]
    # 调用云端模型；temperature=0.7 允许一定创造性
    response = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=messages,
        temperature=0.7
    )
    # 取出助手回复文本并返回
    return response.choices[0].message.content


# --- 能力 2：建议主题行 ---
def suggest_subject(email_content):
    """Suggest a subject line for the email"""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"""
        Please suggest a short appropriate subject 
        line for this email:
        
        {email_content}
        """}
    ]
    response = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=messages,
        temperature=0.7
    )
    return response.choices[0].message.content


# --- 能力 3：摘要并翻译到指定语言（默认 Chinese） ---
def translate_summary(email_content, language="Chinese"):
    """Summarize and translate the email"""
    # user prompt 里嵌入 language，可换成其他目标语言
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"""
        Please summarize this email and translate 
        the summary into {language}:
        
        {email_content}
        """}
    ]
    response = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=messages,
        temperature=0.7
    )
    return response.choices[0].message.content


# --- 能力 4：生成专业回复草稿 ---
def reply_to_email(email_content):
    """Generate a professional reply to the email"""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"""
        Please write a professional reply to this email:
        
        {email_content}
        """}
    ]
    response = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=messages,
        temperature=0.7
    )
    return response.choices[0].message.content


In [ ]:
# ========== 示例邮件正文：改这里就能分析你自己的邮件 ==========

# email_content 保持英文原文：这是发给模型的用户内容，不翻译以免改变任务
# 练习建议：换成你收到的一封真实邮件（注意脱敏）
email_content = """
Hi Emily, 
long time no see! How are you?
I will be visiting London next week. 
Let's meet for dinner if you have time! 
Best regards,
Cindy
"""


In [ ]:
# ========== 串联演示：依次跑四个能力并展示结果 ==========

# 1. 总结电子邮件
print("=" * 50)
print("📧 EMAIL SUMMARY")
print("=" * 50)
# 调用 summarize_email，再用 Markdown 展示
display(Markdown(summarize_email(email_content)))

# 2. 建议主题行
print("=" * 50)
print("📌 SUGGESTED SUBJECT LINE")
print("=" * 50)
display(Markdown(suggest_subject(email_content)))

# 3. 将摘要翻译成中文（第二个参数传 "Chinese"）
print("=" * 50)
print("🌏 CHINESE TRANSLATION")
print("=" * 50)
display(Markdown(translate_summary(email_content, "Chinese")))

# 4. 生成回复草稿
print("=" * 50)
print("✉️ SUGGESTED REPLY")
print("=" * 50)
display(Markdown(reply_to_email(email_content)))
